
# 05 · NBED 위치별 분석 — median · ring 가상이미지· · 구조인자/RDF NMF

한 개의 NBED(nano-beam diffraction) 4D 스캔에서 **위치(probe)별** 로:

1. dm4 불러오기
2. 전체 **median** 회절패턴
3. 여러 **링 영역**의 가상이미지 (구조 대비)
4. 전체 패턴 **PCA** → 서로 다른 ring pattern이 몇 종류인지 (scree/성분)
5. 각 패턴 **radial 분포** I(q)
6. 배경 제거 → **구조인자 φ(q)** (환원 structure factor)
7. 구조인자 **NMF k=4**
8. 각 구조인자 **FFT(sine)** → **RDF G(r)**
9. RDF **NMF k=4**

> 모든 그림은 각 셀에서 PNG로, 그래프 데이터는 CSV로 `SAVE_DIR`에 저장됩니다.
> NMF는 비음수 입력이 필요합니다. φ(q)·G(r)은 부호가 있어 음수는 0으로 클리핑되니(성분이 다소 왜곡),
> `METHOD="pca"`로 바꿔 비교할 수 있게 해두었습니다(기본은 요청대로 `"nmf"`, k=4).


## 1) dm4 불러오기 & 설정

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
Q_UNIT_HINT= "1/nm"     # ★ dm4 단위가 1/nm (0.043888 1/nm/px). 1/A로 두면 q가 10배 틀림
Q_PER_PX   = 0.0120       # 초기 추정(캘리브레이션으로 갱신). 실데이터는 nb1 값 참고
N_JOBS     = 4            # 환원 병렬 코어수. ★메모리 부족(커널 꺼짐)이면 2로 줄이세요(-1/-2=전코어=메모리多)
K          = 4            # NMF 성분 수 (요청: 4)
K_PCA      = 20           # PCA 성분 수(누적분산으로 99%까지 몇 개인지 확인)
N_BINS     = None         # radial bin 수 (None=자동)
METHOD     = "nmf"        # 구조인자/RDF 분해 방법: "nmf"(요청) 또는 "pca"
RINGS_PX   = [(6,12),(12,18),(18,26),(26,36),(36,48),(48,60),(60,72),(72,84),(84,96),(96,108),(108,126),(126,146)]

# --- 중심빔 위치 ---
CENTER        = None       # (cx,cy) 수동 지정. None이면 자동
CENTER_METHOD = "com"      # "com"=중심빔 무게중심(직접빔 강할 때 강건), "friedel"=대칭
CENTER_ZOOM   = 25         # 중심 확대 그림 반경(px). 중심 맞는지 확인용
HOT_THRESHOLD = 8.0        # 고정 hot/dead 픽셀 검출 민감도

# --- 진공(빈 영역) 배경 빼기 ---
# 물질 없는 영역이 PCA/NMF의 1등 분산을 먹지 않도록, 빈 영역 평균 패턴을 전체에서 뺍니다.
VACUUM_SUBTRACT = True      # 빈 영역 평균 빼기 on/off
MATERIAL_MASK   = True      # 빈(진공) 위치를 분석에서 제외(산란 세기 기준)
EMPTY_ROWS      = 10        # 아래쪽 이만큼의 행이 빈 영역(4D 스캔 기준)
EMPTY_ROI       = None      # (y0,y1,x0,x1)로 빈 영역 직접 지정(있으면 EMPTY_ROWS 무시)

# --- q 캘리브레이션 / 1st peak ---
CALIBRATE  = False         # 혼합 Li 화합물이라 1st peak를 특정값에 강제하면 안 됨 → 메타데이터 그대로.
                           # (아는 기준 거리가 있을 때만 True + R_TARGET)
                           # False: 메타데이터 q_per_px 그대로 써서 '자연' 위치 확인
R_TARGET   = 2.0           # 이 시료의 1st peak(≈1.9~2.2 Å). SiO2가 아니면 1.61 아님!
FIRST_WIN  = (1.6, 2.5)    # 1st peak 탐색창 (1.9~2.2 포함하도록)
CALIB_WIN  = (1.5, 3.0)    # 캘리브레이션용 피크 탐색창(넓게)

# 배경 제거용 조성(원자 산란인자). 시료가 SiO2가 아니면 실제 조성으로 바꾸세요.
# 알려진 Li 화합물 1차 최근접 거리(Å, 결정 근사) — RDF에 참고선으로 겹침(비정질은 다소 이동)
LI_REF = {"Li-F (LiF) 2.01": 2.01, "Li-O (Li2O) 2.00": 2.00, "Li-N (Li3N) 1.94": 1.94,
          "Li-S (Li2S) 2.48": 2.48, "anion-anion ~2.85": 2.85}   # 후보 5개 최근접이웃
# 배경 제거용 조성(원자 산란인자). 실제 시료 조성으로 바꾸세요(예: LiF -> {"Li":1,"F":1}).
CFG = fds.RDFConfig(composition={"Li":1,"O":1}, q_int_min=0.20, q_int_max=1.50,
                    r_min=1.10, r_max=8.0, dr=0.02, damping="lorch")

def make_nbed_cube(scan=(40,40), dp=(96,96), empty_rows=10, seed=0):
    '''합성 데모: 스캔 위쪽은 3개 영역의 서로 다른 ring pattern(반경), **아래 empty_rows 행은
    빈 영역(진공: 중심빔만)**. 진공 빼기가 아래 영역을 ~0으로 만드는지 확인용.'''
    rng = np.random.default_rng(seed); Sy,Sx = scan; H,W = dp
    yy,xx = np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    def ring(r0,a=1.0,s=3.0): return a*np.exp(-(rr-r0)**2/(2*s**2))
    beam = 6*np.exp(-rr**2/(2*2.0**2))
    phases = [ring(20)+0.4*ring(34), ring(26)+0.3*ring(40), ring(16)+0.5*ring(30)]
    cube = np.empty((Sy,Sx,H,W), np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            if iy >= Sy-empty_rows:                       # 빈 영역: 중심빔 + 노이즈만
                cube[iy,ix] = 0.9*beam + 0.8*rng.standard_normal((H,W))
            else:
                ph = 0 if ix < Sx//3 else (1 if ix < 2*Sx//3 else 2)
                cube[iy,ix] = beam + phases[ph]*(1+0.1*rng.standard_normal()) \
                              + 0.8*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube = fds.from_array(make_nbed_cube(empty_rows=EMPTY_ROWS), q_per_px=Q_PER_PX, name="synthetic-NBED")
else:
    cube = fds.load(DM4_PATH, Q_UNIT_HINT)
    print("raw loaded shape:", cube.data.shape, "(ndim", cube.ndim, ")")
    if cube.ndim < 3:
        raise ValueError(
            f"loaded data is {cube.ndim}D {cube.data.shape} — a single pattern, not a scan. "
            "This dm4 likely stores the 3D/4D data in another dataset. The reader now searches "
            "datasets automatically; if you still see 2D, load the correct array yourself and "
            "wrap it:  cube = fds.from_array(arr, q_per_px=...).")
    if DET_BIN > 1:
        cube = fds.bin_cube_detector(cube, DET_BIN)      # works for 3D stack or 4D scan
scan = cube.scan_shape; dp = cube.dp_shape
b = 1 if USE_SYNTHETIC else max(DET_BIN,1)
rings = [(ri/b, ro/b) for ri,ro in RINGS_PX]

# 스캔이 4D면 (Ry,Rx) 2D 맵, 3D 스택이면 (n,) 1D → 표시용으로 near-square 2D 맵으로 재배열
import math
def _mapshape(n):
    r = int(math.sqrt(n))
    while r > 1 and n % r: r -= 1
    return (r, n//r) if r > 1 else (1, n)
MAP = tuple(scan) if len(scan) == 2 else _mapshape(int(np.prod(scan)))
def as_map(vec):
    v = np.asarray(vec, float).ravel()
    m = np.full(int(np.prod(MAP)), np.nan); m[:min(v.size, m.size)] = v[:m.size]
    return m.reshape(MAP)
print("USE_SYNTHETIC =", USE_SYNTHETIC, "| cube:", cube.shape, "| scan:", scan,
      "| dp:", dp, "| display map:", MAP)

# 저장 위치 & 헬퍼
SAVE_DIR = (os.path.dirname(DM4_PATH)+"/nb5_outputs" if not USE_SYNTHETIC else "nb5_outputs")
os.makedirs(SAVE_DIR, exist_ok=True)
def save(fig, name):
    p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name, header, rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f:
        w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:", p)
print("outputs ->", os.path.abspath(SAVE_DIR))



## 2) 전체 median 회절패턴 + 중심빔 점검

평균 대신 **median** — X선 히트·hot frame·간헐 Bragg 스팟 같은 이상치를 걸러낸 "대표" 패턴.
중심은 `CENTER`(수동) 또는 `CENTER_METHOD`("com"=중심빔 무게중심, "friedel"=대칭)로 잡습니다.
**오른쪽 방위각 프로파일(중심 점검)** — 링이 **날카로운 하나의 봉우리**면 중심이 정확, 넓게 퍼지거나
두 갈래면 중심이 틀린 것 → `CENTER`로 직접 지정하세요.


In [ ]:

med = fds.median_pattern(cube)
stop = fds.beam_stopper_mask(med)
if CENTER is not None:
    cx, cy = CENTER; how = "manual"
elif CENTER_METHOD == "com":
    cx, cy = fds.center_of_mass(med, threshold=0.3); how = "com(threshold=0.3)"
else:
    (cx, cy), _ = fds.find_center(med, stop); how = "friedel"
print(f"center: ({cx:.1f}, {cy:.1f})  [{how}]")

# 중심 교차검증: COM vs Friedel vs 기하중심. 서로 5px 이상 다르면 의심.
try:
    (fcx, fcy), fried = fds.find_center(med, stop)
    print(f"  cross-check Friedel: ({fcx:.1f}, {fcy:.1f})  (Friedel score {fried:.2f}; 1.0=완벽 대칭)")
except Exception as e:
    fcx = fcy = None; print("  Friedel cross-check failed:", e)
gcx, gcy = dp[1]/2.0, dp[0]/2.0
print(f"  geometric center: ({gcx:.1f}, {gcy:.1f}) | used-center offset = {np.hypot(cx-gcx, cy-gcy):.1f} px")
if fcx is not None and np.hypot(cx-fcx, cy-fcy) > 5:
    print("  ⚠ COM와 Friedel이 5px 이상 다릅니다 — 아래 방위각 프로파일로 확인하고, 더 날카로운 쪽을 CENTER로 지정하세요.")

# 중심 점검용 방위각 적분(px 단위)
qd, Id = fds.azimuthal_integrate(med, (cx, cy), q_per_px=1.0)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
im0 = ax[0].imshow(med, cmap="magma"); ax[0].plot(cx, cy, "c+", ms=12, mew=2)
ax[0].set_title("median pattern"); plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(np.log1p(med), cmap="magma"); ax[1].plot(cx, cy, "c+", ms=12, mew=2)
th = np.linspace(0, 2*np.pi, 100)
for rad in (qd[np.nanargmax(np.where(qd>3, Id, -np.inf))],):   # 첫 링 반경
    ax[1].plot(cx+rad*np.cos(th), cy+rad*np.sin(th), "c-", lw=0.6)
ax[1].set_title("median (log) + center & 1st ring"); plt.colorbar(im1, ax=ax[1], fraction=0.046)
ax[2].plot(qd, Id, "-"); ax[2].set_xlabel("radius (px)"); ax[2].set_ylabel("azimuthal mean")
ax[2].set_title("center check: sharp single ring = good center")
plt.tight_layout(); save(fig, "02_median_pattern"); plt.show()
np.save(os.path.join(SAVE_DIR, "02_median_pattern.npy"), med); print("saved: 02_median_pattern.npy")

# MAX 프로젝션(가장 강한 신호) + 물질 마스크(빈 영역 제외)
mx = cube.max_dp()
if MATERIAL_MASK and not USE_SYNTHETIC and len(scan) == 2:
    emp = None
    if EMPTY_ROI is not None:
        emp = np.zeros(scan, bool); y0,y1,x0,x1 = EMPTY_ROI; emp[y0:y1, x0:x1] = True
    elif EMPTY_ROWS:
        emp = np.zeros(scan, bool); emp[max(0, scan[0]-EMPTY_ROWS):, :] = True
    material = fds.material_mask(cube, center=(cx, cy), empty_mask=emp)
else:
    material = np.ones(scan, bool)
KEEP = material.ravel()
def scatter(vec, fill=np.nan):
    out = np.full(KEEP.size, fill, float); out[KEEP] = np.asarray(vec, float).ravel(); return out.reshape(scan)
print(f"material positions: {int(KEEP.sum())}/{KEEP.size} ({100*KEEP.mean():.0f}%)")
figm, axm = plt.subplots(1, 2, figsize=(9, 4))
axm[0].imshow(np.log1p(mx), cmap="magma"); axm[0].set_title("MAX projection (strongest signal)"); axm[0].axis("off")
axm[1].imshow(material, cmap="gray"); axm[1].set_title("material mask (white = analyzed)"); axm[1].axis("off")
plt.tight_layout(); save(figm, "02_max_material"); plt.show()
np.save(os.path.join(SAVE_DIR, "02_max_projection.npy"), np.asarray(mx))


## 2c) NBD 전처리 점검 — 빔 wander · 고정 bad pixel · 중심 확대

넓은 영역에서 **직접빔 이동(wander)**, 매 프레임 같은 위치의 **hot/dead 픽셀**, 그리고 **중심이 실제 빔에
얹혀 있는지**(확대) 확인합니다. wander가 크면 §5~8 위치별 RDF가 빔 움직임에 오염될 수 있습니다.

In [ ]:
comx, comy = fds.center_of_mass_map(cube, normalize=True)
shift = np.hypot(np.asarray(comx), np.asarray(comy))
badmap = fds.bad_pixel_map(np.asarray(cube.max_dp(),float), hot_threshold=HOT_THRESHOLD)
print(f'beam wander (material): median {np.nanmedian(shift[material]):.2f}px, max {np.nanmax(shift[material]):.2f}px')
print(f'bad (hot/dead) pixels: {int(badmap.sum())} ({100*badmap.mean():.2f}% of detector)')
ZOOM=int(min(CENTER_ZOOM, dp[0]//2, dp[1]//2))
yy0,yy1=max(0,int(round(cy))-ZOOM),min(dp[0],int(round(cy))+ZOOM); xx0,xx1=max(0,int(round(cx))-ZOOM),min(dp[1],int(round(cx))+ZOOM)
fig,ax=plt.subplots(1,3,figsize=(14,4))
im=ax[0].imshow(np.where(material,shift,np.nan),cmap='viridis'); ax[0].set_title('beam wander |shift| (px)'); ax[0].axis('off'); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[1].imshow(badmap,cmap='gray'); ax[1].set_title(f'bad-pixel map ({int(badmap.sum())})'); ax[1].axis('off')
ax[2].imshow(np.log1p(med[yy0:yy1,xx0:xx1]),cmap='magma',extent=[xx0,xx1,yy1,yy0])
ax[2].axhline(cy,color='cyan',lw=0.7,ls='--'); ax[2].axvline(cx,color='cyan',lw=0.7,ls='--'); ax[2].plot(cx,cy,'c+',ms=18,mew=2)
ax[2].set_title(f'center zoom (+/-{ZOOM}px): + on beam?'); ax[2].set_xlabel('det x (px)'); ax[2].set_ylabel('det y (px)')
plt.tight_layout(); save(fig,'02c_preprocess'); plt.show()



## 2b) 진공(빈 영역) 배경 빼기 — PCA/NMF가 '물질 없는 곳'에 홀리지 않게

빈 영역(예: 아래 `EMPTY_ROWS` 행)은 물질과 너무 달라서 PCA/NMF의 **1등 성분**을 다 차지해버립니다. 그 영역
평균 패턴(진공+검출기 배경+직접빔)을 **전체에서 빼면** 빈 영역이 ~0이 되어, 분해가 **물질 내부 구조 차이**에
집중합니다. 이후 3~9단계는 이 배경 뺀 큐브를 씁니다.


In [ ]:

if VACUUM_SUBTRACT and len(scan) == 2:
    Ry, Rx = scan
    empty = np.zeros(scan, bool)
    if EMPTY_ROI is not None:
        y0,y1,x0,x1 = EMPTY_ROI; empty[y0:y1, x0:x1] = True; y_line = y0
    else:
        y_line = max(0, Ry - EMPTY_ROWS); empty[y_line:, :] = True
    vac = fds.average_pattern(cube, empty)
    ri0, ro0 = min(dp)/6.0, min(dp)/3.0
    before = np.asarray(fds.annular_dark_field(cube, center=(cx,cy), r_inner=ri0, r_outer=ro0))
    cube = fds.subtract_reference(cube, vac)                 # ★ 배경 뺀 큐브로 교체
    after = np.asarray(fds.annular_dark_field(cube, center=(cx,cy), r_inner=ri0, r_outer=ro0))
    print(f"vacuum subtracted from {int(empty.sum())} empty positions (rows >= {y_line})")

    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    ax[0].imshow(np.log1p(np.clip(vac,0,None)), cmap="magma"); ax[0].set_title("vacuum reference (empty avg)")
    ax[0].axis("off")
    im1=ax[1].imshow(before, cmap="viridis"); ax[1].set_title("ring image BEFORE"); ax[1].axis("off")
    im2=ax[2].imshow(after, cmap="viridis"); ax[2].set_title("ring image AFTER (empty ~0)"); ax[2].axis("off")
    for a in (ax[1], ax[2]): a.axhline(y_line-0.5, color="r", lw=1)
    plt.tight_layout(); save(fig, "02b_vacuum_subtraction"); plt.show()
    np.save(os.path.join(SAVE_DIR, "02b_vacuum_reference.npy"), np.asarray(vac))
    print("saved: 02b_vacuum_reference.npy")
else:
    print("vacuum subtraction: OFF (or scan not 2D)")



## 3) 링 영역별 가상이미지

여러 반경 링(`RINGS_PX`)을 환형 검출기로 써서 각 링의 산란 세기를 스캔 상에 매핑합니다. 링마다 다른
상/구조가 다른 대비로 나타납니다.


In [ ]:

nR = len(rings)
fig, ax = plt.subplots(1, nR+1, figsize=(3.0*(nR+1), 3.2))
ax[0].imshow(np.log1p(med), cmap="magma")
for ri,ro in rings:
    th=np.linspace(0,2*np.pi,100)
    for rad in (ri,ro): ax[0].plot(cx+rad*np.cos(th), cy+rad*np.sin(th), "c-", lw=0.6)
ax[0].set_title("rings on median"); ax[0].axis("off")
vimgs=[]
for j,(ri,ro) in enumerate(rings):
    vi = np.asarray(fds.annular_dark_field(cube, center=(cx,cy), r_inner=ri, r_outer=ro))
    vimgs.append(vi)
    im=ax[j+1].imshow(as_map(vi), cmap="viridis"); ax[j+1].set_title(f"ring {int(ri)}-{int(ro)} px", fontsize=9)
    ax[j+1].axis("off")
plt.tight_layout(); save(fig, "03_ring_virtual_images"); plt.show()
# 저장: 각 링 가상이미지(스캔 평탄화 컬럼)
save_csv("03_ring_virtual_images",
         ["scan_index"]+[f"ring_{int(ri)}_{int(ro)}" for ri,ro in rings],
         [[i]+[f"{v.ravel()[i]:.6g}" for v in vimgs] for i in range(vimgs[0].size)])



## 5) 위치별 radial 분포 I(q) + q 캘리브레이션

먼저 **메타데이터 q_per_px 그대로** median 패턴의 1st peak가 어디 오는지 봅니다. 그게 이미 1.9~2.2 Å면
캘리브레이션 불필요(`CALIBRATE=False`). 어긋나면(단위 오표기 등) `R_TARGET`(이 시료 ≈2.0 Å)에 맞춰 보정합니다.
> **주의**: 캘리브레이션은 피크를 R_TARGET에 '강제'로 맞추는 것이라, R_TARGET을 실제 시료값으로 정확히
> 넣어야 합니다. 모르면 `CALIBRATE=False`로 두고 메타데이터가 주는 자연 위치를 신뢰하세요.

> **메모리(커널이 꺼질 때)**: §5~10은 위치마다 계산합니다. 꺼지면 ① `DET_BIN`을 키우고(검출기 메모리 ¼로), ② `N_JOBS`를 2로 낮추고, ③ 스캔이 매우 크면 관심 영역만 잘라서 진행하세요. (radial_profiles는 이제 블록 단위 float32라 예전보다 훨씬 가볍습니다.)

In [ ]:

def _r1(qpp, win=CALIB_WIN):
    rr = fds.pattern_to_rdf(med, qpp, CFG, center=(cx,cy),
                            center_beam_radius=max(1, int(CFG.q_int_min/qpp)))
    r1,_ = fds.first_peak_position(rr.r, rr.Gr, *win); return r1

QPP_meta = cube.calibration.q_per_px or Q_PER_PX
print(f"metadata q_per_px = {QPP_meta:.5g} 1/A/px  ->  1st peak at {_r1(QPP_meta):.3f} Å"
      f"  (search {CALIB_WIN} Å)")

QPP = QPP_meta
if CALIBRATE and not USE_SYNTHETIC:
    for _ in range(8):
        ri=_r1(QPP)
        if not np.isfinite(ri) or ri<=0 or abs(ri-R_TARGET)<0.003: break
        QPP *= ri/R_TARGET
    print(f"[calib] q_per_px -> {QPP:.5g}  (1st peak -> {_r1(QPP):.3f} Å, target {R_TARGET})")
else:
    print(f"[calib] no forcing — using q_per_px = {QPP:.5g}  (1st peak {_r1(QPP):.3f} Å)")

profiles, r_px = fds.radial_profiles(cube, (cx,cy), n_bins=N_BINS, normalize=False)
q = r_px * QPP
# --- 캘리브레이션 체인 + FSDP(배경제거) 검출: 링이 실제 있는지/신뢰도 ---
qpp_unbin = QPP/max(DET_BIN if not USE_SYNTHETIC else 1,1)
Imean = np.nanmean(profiles[KEEP], 0) if KEEP.any() else profiles.mean(0)
q_ring, ring_conf = fds.find_fsdp(q, Imean, q_lo=0.30, q_hi=0.95)
print(f'[calibration chain] dm ~ {qpp_unbin*10:.5f} 1/nm/px -> /10 = {qpp_unbin:.6f} 1/A x DET_BIN = QPP {QPP:.6f} 1/A/px')
print(f'  FSDP: q_ring={q_ring:.3f} 1/A (conf {ring_conf:.1f}) | d=1/q={1/q_ring:.2f} | nn~1.23/q={1.23/q_ring:.2f} A (물질평균)')
print('  '+('\u2713 링 신뢰 가능' if ring_conf>=5 else f'\u26a0 conf {ring_conf:.1f}<5: 뚜렷한 링 없음 \u2192 메타데이터 calibration 신뢰, nn 판독 주의'))
print("profiles:", profiles.shape, "| q range:", round(q.min(),3), "..", round(q.max(),3), "1/A")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].plot(q, profiles.mean(0), "k-", lw=2, label="mean")
for i in np.linspace(0, len(profiles)-1, 6).astype(int):
    ax[0].plot(q, profiles[i], lw=0.6, alpha=0.6)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("I(q)"); ax[0].set_title("radial profiles (mean + samples)"); ax[0].legend()
im=ax[1].imshow(profiles, aspect="auto", cmap="viridis",
                extent=[q.min(), q.max(), len(profiles), 0])
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("scan position index"); ax[1].set_title("I(q) — all positions")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "05_radial_profiles"); plt.show()
save_csv("05_radial_mean", ["q_invA","I_mean"],
         [[f"{q[i]:.5f}", f"{profiles.mean(0)[i]:.6g}"] for i in range(len(q))])



## 6) 배경 제거 → 구조인자 φ(q)

각 I(q)를 환원(scale N 자동적합, 원자 산란인자로 배경 제거)하여 **구조인자 φ(q)=q(S(q)−1)** 로 만듭니다.
위치가 많으면 병렬(`N_JOBS`)로 수 분 걸릴 수 있습니다.


In [ ]:

red = fds.reduce_profiles(profiles, q, CFG, n_jobs=N_JOBS, progress=True)
qf, phi, rr, Gr, ok = red["q"], red["phi"], red["r"], red["Gr"], red["ok"] & KEEP
print("reduced ok:", int(ok.sum()), "/", len(ok), "| phi:", phi.shape, "| Gr:", Gr.shape)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].axhline(0, color="0.8", lw=0.8)
ax[0].plot(qf, np.nanmean(phi[ok], 0), "k-", lw=2, label="mean")
for i in np.where(ok)[0][::max(1, ok.sum()//6)][:6]:
    ax[0].plot(qf, phi[i], lw=0.6, alpha=0.6)
ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("φ(q)"); ax[0].set_title("structure factor φ(q)"); ax[0].legend()
im=ax[1].imshow(phi[ok], aspect="auto", cmap="coolwarm",
                extent=[qf.min(), qf.max(), int(ok.sum()), 0],
                vmin=-np.nanpercentile(np.abs(phi[ok]),98), vmax=np.nanpercentile(np.abs(phi[ok]),98))
ax[1].set_xlabel("q (1/Å)"); ax[1].set_ylabel("position (ok only)"); ax[1].set_title("φ(q) — all positions")
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "06_structure_factor"); plt.show()
save_csv("06_structure_factor_mean", ["q_invA","phi_mean"],
         [[f"{qf[i]:.5f}", f"{np.nanmean(phi[ok],0)[i]:.6g}"] for i in range(len(qf))])



### 6b) RDF 품질 점검 (QC) — "잘 만들어졌나?"

물질 영역 **평균 패턴**의 RDF를 객관 지표로 점검합니다: 1st peak 위치, r_min 아래 평탄도, φ(q)가 고q에서
**램프(발산)하지 않는지**. `verdict=good` 이고 φ가 안 튀면 신뢰. `q_int_max`가 신호보다 짧으면 RDF가
거칠어지니(피크폭↑), §5 I(q)에서 신호가 있는 q까지 `CFG.q_int_max`를 올리세요.


In [ ]:

mat_pat = fds.average_pattern(cube, material)               # 물질 영역 평균 패턴
res = fds.pattern_to_rdf(mat_pat, QPP, CFG, center=(cx,cy),
                         center_beam_radius=max(1, int(CFG.q_int_min/QPP)))
Q = fds.rdf_quality(res, expected_first_peak=R_TARGET, first_win=FIRST_WIN)
print("q_int_max =", CFG.q_int_max, "1/Å  | data max q =", round(float(q.max()),2),
      "1/Å  (신호가 더 높은 q까지 있으면 q_int_max를 올려 RDF를 더 선명하게)")
print(f"1st peak r1 = {Q['first_peak_r']:.3f} Å (target {R_TARGET}, offset {Q['first_peak_offset']:+.3f})")
print(f"low-r RMS/peak = {Q['low_r_rms']:.3f} | φ high-q drift = {Q['phi_highq_drift']:+.2f}"
      f" (|drift|<1.5 OK) | qmax = {Q['qmax']:.2f}")
print("flags:", Q["flags"], "->  VERDICT:", Q["verdict"].upper())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].axhline(0,color="0.8",lw=.8); ax[0].plot(res.q_reduced, res.phi, "k-")
ax[0].axhline(0,color="0.8"); ax[0].set_xlabel("q (1/Å)"); ax[0].set_ylabel("φ(q)")
ax[0].set_title("φ(q) — should oscillate & decay, not ramp")
ax[1].axhline(0,color="0.8",lw=.8); ax[1].plot(res.r, res.Gr, "b-")
ax[1].axvline(Q["first_peak_r"], color="r", ls="--", label=f"1st peak {Q['first_peak_r']:.2f} Å")
for lbl,d in LI_REF.items():                         # 알려진 Li 화합물 거리 참고선
    ax[1].axvline(d, color="0.6", ls=":", lw=1); ax[1].text(d, ax[1].get_ylim()[1]*0.92,
        lbl.split()[0], rotation=90, fontsize=6, va="top", ha="right", color="0.4")
ax[1].set_xlim(0,8); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)")
ax[1].set_title(f"mean-material RDF vs Li-compound distances — verdict: {Q['verdict']}"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig, "06b_rdf_qc"); plt.show()
print("참고 거리(Å):", {k: v for k,v in LI_REF.items()})



## 7) 구조인자 NMF (k=4)

구조인자 스택을 `k=4` 성분으로 분해 → **4개의 대표 구조인자**(성분)와 각 위치의 **분율(loading 맵)**.
서로 다른 링 패턴/상이 성분으로 분리됩니다. (φ는 부호가 있어 음수는 0으로 클리핑; `METHOD="pca"`로 비교 가능)


In [ ]:

def decompose_and_map(stack, ok, k, method, xaxis):
    dpc = fds.decompose_profiles(stack[ok], n_components=k, method=method, x=xaxis)
    maps = np.full((k, int(np.prod(scan))), np.nan); idx=np.where(ok)[0]
    for i in range(k): maps[i, idx] = dpc.fractions[:, i]
    return dpc, maps.reshape(k, *scan)

dp_sf, sf_maps = decompose_and_map(phi, ok, K, METHOD, qf)
fig, ax = plt.subplots(2, K, figsize=(3.2*K, 6))
for i in range(K):
    ax[0,i].axhline(0, color="0.85", lw=0.8); ax[0,i].plot(qf, dp_sf.components[i], lw=1.4)
    ax[0,i].set_title(f"SF comp {i+1}", fontsize=9); ax[0,i].set_xlabel("q (1/Å)")
    im=ax[1,i].imshow(as_map(sf_maps[i]), cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9)
    ax[1,i].axis("off")
fig.suptitle(f"structure factor {METHOD.upper()} (k={K})", y=1.02)
plt.tight_layout(); save(fig, "07_structure_factor_nmf"); plt.show()
save_csv("07_sf_components", ["q_invA"]+[f"comp{i+1}" for i in range(K)],
         [[f"{qf[j]:.5f}"]+[f"{dp_sf.components[i][j]:.6g}" for i in range(K)] for j in range(len(qf))])



## 8) 구조인자 FFT(sine) → RDF G(r)

각 φ(q)를 사인 변환하여 위치별 **RDF G(r)** 을 얻습니다(환원 단계에서 함께 계산됨).


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ax[0].axhline(0, color="0.8", lw=0.8)
ax[0].plot(rr, np.nanmean(Gr[ok], 0), "k-", lw=2, label="mean")
for i in np.where(ok)[0][::max(1, ok.sum()//6)][:6]:
    ax[0].plot(rr, Gr[i], lw=0.6, alpha=0.6)
ax[0].set_xlim(0, 8); ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("G(r)")
ax[0].set_title("RDF G(r)"); ax[0].legend()
im=ax[1].imshow(Gr[ok], aspect="auto", cmap="coolwarm", extent=[rr.min(), rr.max(), int(ok.sum()), 0],
                vmin=-np.nanpercentile(np.abs(Gr[ok]),98), vmax=np.nanpercentile(np.abs(Gr[ok]),98))
ax[1].set_xlim(0, 8); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("position (ok only)")
ax[1].set_title("G(r) — all positions"); plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); save(fig, "08_rdf"); plt.show()
save_csv("08_rdf_mean", ["r_A","Gr_mean"],
         [[f"{rr[i]:.4f}", f"{np.nanmean(Gr[ok],0)[i]:.6g}"] for i in range(len(rr))])



## 9) RDF NMF (k=4)

RDF 스택을 `k=4` 성분으로 분해 → **4개의 대표 RDF**와 위치별 분율 맵. 구조적으로 다른 국소 배열이 성분으로
나뉩니다. (G(r)도 부호 있음 → 음수 클리핑 주의; `METHOD="pca"` 비교 가능)


In [ ]:

dp_rdf, rdf_maps = decompose_and_map(Gr, ok, K, METHOD, rr)
fig, ax = plt.subplots(2, K, figsize=(3.2*K, 6))
for i in range(K):
    ax[0,i].axhline(0, color="0.85", lw=0.8); ax[0,i].plot(rr, dp_rdf.components[i], lw=1.4)
    ax[0,i].set_xlim(0, 8); ax[0,i].set_title(f"RDF comp {i+1}", fontsize=9); ax[0,i].set_xlabel("r (Å)")
    im=ax[1,i].imshow(as_map(rdf_maps[i]), cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9)
    ax[1,i].axis("off")
fig.suptitle(f"RDF {METHOD.upper()} (k={K})", y=1.02)
plt.tight_layout(); save(fig, "09_rdf_nmf"); plt.show()
save_csv("09_rdf_components", ["r_A"]+[f"comp{i+1}" for i in range(K)],
         [[f"{rr[j]:.4f}"]+[f"{dp_rdf.components[i][j]:.6g}" for i in range(K)] for j in range(len(rr))])
print("\nAll outputs in:", os.path.abspath(SAVE_DIR))


## 10) 후보 Li 화합물로 위치별 supervised 언믹싱 (NNLS) → 화합물 함량 맵

§9의 임의 NMF 대신, **아는 5개 화합물 signature**(결정 거리로 합성한 RDF)를 기저로 **각 위치의 RDF**를
맞춰 **화합물별 함량 지도**를 만듭니다. 없는 화합물은 함량 ≈ 0(그 맵이 전체적으로 어두움). **참조 분리도**로
이 분해능에서 구별 가능한 쌍을 먼저 확인하세요(LiF/Li2O/Li3N은 겹침, Li2S·Li2CO3는 분리). `fit residual`
맵이 밝은 위치는 후보로 설명이 안 되는 곳(다른 상/노이즈).

In [ ]:
CANDIDATES = ['LiF','Li2O','Li3N','Li2CO3','Li2S']
qmax_eff = min(CFG.q_int_max, float(q.max()))
REF_SIGMA = float(np.clip(0.5/max(qmax_eff,0.4), 0.25, 0.8))
refs = fds.build_references(rr, compounds=CANDIDATES, sigma=REF_SIGMA)
nms,C=fds.reference_degeneracy(refs)
print(f'REF_SIGMA={REF_SIGMA:.2f} A | 참조 분리도(1=구별불가):')
print('         '+' '.join(f'{n:>7s}' for n in nms))
for i,n in enumerate(nms): print(f'{n:>7s} '+' '.join(f'{C[i,j]:7.2f}' for j in range(len(nms))))

names, AB, resid = fds.unmix_nnls(Gr, refs, r=rr, r_range=(CFG.r_min, 6.0))
AB=np.asarray(AB,float); resid=np.asarray(resid,float)
AB[~ok]=np.nan; resid[~ok]=np.nan
print('mean abundance (ok positions):', {n:round(float(np.nanmean(AB[ok,j])),2) for j,n in enumerate(names)})

nC=len(names)
fig,ax=plt.subplots(1,nC+1,figsize=(3.1*(nC+1),3.4))
for j,c in enumerate(names):
    im=ax[j].imshow(as_map(AB[:,j]), cmap='viridis', vmin=0, vmax=1); ax[j].set_title(c,fontsize=10); ax[j].axis('off'); plt.colorbar(im,ax=ax[j],fraction=0.046)
im=ax[nC].imshow(as_map(resid), cmap='magma'); ax[nC].set_title('fit residual',fontsize=10); ax[nC].axis('off'); plt.colorbar(im,ax=ax[nC],fraction=0.046)
fig.suptitle(f'per-position compound abundance (NNLS, σ={REF_SIGMA:.2f} Å)', y=1.03)
plt.tight_layout(); save(fig,'10_compound_abundance_maps'); plt.show()

# 지배 화합물(argmax) — 신뢰(잔차 낮은) 위치만
dom=np.full(len(ok),-1,int)
if ok.any():
    thr=np.nanpercentile(resid[ok],80); good=ok & (resid<thr)
    if good.any(): dom[good]=np.nanargmax(AB[good],1)
fig,ax=plt.subplots(1,1,figsize=(5.2,3.6))
im=ax.imshow(as_map(np.where(dom>=0,dom,np.nan)), cmap='tab10', vmin=0, vmax=nC-1)
ax.set_title('dominant compound (argmax, good fits)'); ax.axis('off')
cb=plt.colorbar(im,ax=ax,ticks=range(nC)); cb.ax.set_yticklabels(names)
plt.tight_layout(); save(fig,'10_dominant_compound_map'); plt.show()
save_csv('10_compound_abundance', ['scan_index']+names+['fit_res'],
         [[i]+[f'{AB[i,j]:.4f}' for j in range(nC)]+[f'{resid[i]:.4f}'] for i in range(len(ok))])
print('\nAll outputs in:', os.path.abspath(SAVE_DIR))



**정리** — 한 NBED 스캔에서 위치별로 (2) median, (3) 링 가상이미지, (4) 위치별 화합물 언믹싱, (5) radial I(q),
(6) 구조인자 φ(q), (7) φ(q) NMF k=4, (8) FFT→RDF, (9) RDF NMF k=4 를 계산·저장합니다.
- 위치 수가 많으면 (6)이 느립니다 → `N_JOBS`(코어), `DET_BIN`(비닝)으로 조절.
- φ(q)·G(r)은 부호가 있어 NMF에서 음수가 클리핑됩니다. 성분 해석이 이상하면 `METHOD="pca"`와 비교하세요.
- (4) scree의 elbow와 (7)·(9)의 성분 수(k)를 데이터에 맞춰 조정하세요.
